# Main script for the Robust KalmanNet project

Import of important modules:

In [1]:
# Select the SS model you would like to evaluate with Robust KalmanNet:
# 0 - Synthetic NL model
# 1 - Discrete Time Lorentz Attractor

SELECTED_MODEL = 0

In [2]:
import torch
from torch.nn.functional import mse_loss
import Simulations.config as config
import matplotlib.pyplot as plt
import math
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
import pandas as pd

from datetime import datetime

assert SELECTED_MODEL <= 1, f"The selected model {SELECTED_MODEL} does not exist!"
if SELECTED_MODEL == 0:
    from Simulations.Synthetic_NL_model.parameters import Q_structure, R_structure,m,n,m1x_0,m2x_0, \
        f,h
elif SELECTED_MODEL == 1:
    from Simulations.Lorenz_Atractor.parameters import m1x_0, m2x_0, m, n,\
    f_gen, f, f_nobatch, h, h_nobatch, hRotate_nobatch, hRotate, H_Rotate, H_Rotate_inv, Q_structure, R_structure

from Simulations.Extended_sysmdl import SystemModel
from Simulations.utils import DataGen
from RobustKalmanPY.robust_kalman import RobustKalman

# KalmanNet
from Pipelines.Pipeline_EKF import Pipeline_EKF
from KNet.KalmanNet_nn import KalmanNetNN

# tqdm for jupyter notebook
from tqdm.notebook import tqdm

# Just so that some warnings are ignored
import warnings
warnings.filterwarnings('ignore')

/opt/homebrew/Caskroom/miniconda/base/envs/kalman_net/lib/python3.9/site-packages/torch/torch_version.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]
/opt/homebrew/Caskroom/miniconda/base/envs/kalman_net/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Settings

In [ ]:
args = config.general_settings()

# Dataset parameters
# N is the number of sequences to be generated

# dataset per testare reali performance 
args.N_E = 200 #length of training dataset
args.N_CV = 50 #length of validation dataset 
args.N_T = 100 #length of test dataset 
args.T = 100 #input sequence length
args.T_test = 100 #input test sequence length
args.n_batch = 10 #input batch size for training (default: 20)

# #valori per testare che tutto funzioni
# args.N_E = 10 #length of training dataset
# args.N_CV = 5 #length of validation dataset 
# args.N_T = 5 #length of test dataset 
# args.T = 50 #input sequence length
# args.T_test = 50 #input test sequence length
# args.n_batch = 3 #input batch size for training (default: 20)


# Training parameters
args.use_cuda = False # use GPU or not (True = use GPU)
args.n_steps =  200 #number of training steps, i.e. number of epochs (default: 1000)
args.lr = 1e-3 #learning rate (default: 1e-3)
args.wd = 1e-3 #weight decay (default: 1e-4)
device = torch.device('cpu')

path_results = 'KNet/'
if SELECTED_MODEL == 0:
    DatafolderName = 'Simulations/Synthetic_NL_model/data' + '/'
else:
    DatafolderName = 'Simulations/Lorenz_Atractor/data' + '/'

# Noise q and r
r2 = torch.tensor([0.1]) # [100, 10, 1, 0.1, 0.01]
vdB = -20 # ratio v=q2/r2
v = 10**(vdB/10) #vdb = 10*log(q2/r2)
q2 = torch.mul(v,r2) #see paper pag. 8 
print("1/r2 [dB]: ", 10 * torch.log10(1/r2[0]))
print("1/q2 [dB]: ", 10 * torch.log10(1/q2[0]))

Q = q2[0] * Q_structure  #defining the transition matrix noise
R = r2[0] * R_structure  #defining the output matrix noise

# Filenames to load the data
if SELECTED_MODEL == 0:
    traj_resultName = ['traj_synNL_T100.pt']
    dataFileName = ['data_synNL_T100.pt']
else:
    traj_resultName = ['traj_lorDT_rq1030_T100.pt']
    dataFileName = ['data_lor_v20_rq1030_T100.pt'] 

1/r2 [dB]:  tensor(10.)
1/q2 [dB]:  tensor(30.)


In [4]:
# model check
print("m1x_0:\n", m1x_0)
print("m2x_0:\n", m2x_0)

# If Q and R are tensors, you can view the dimension and specific numerical values：
print("Q shape:", Q.shape)
print("Q value:\n", Q)

print("R shape:", R.shape)
print("R value:\n", R)


m1x_0:
 tensor([[1.],
        [1.]])
m2x_0:
 tensor([[0., 0.],
        [0., 0.]])
Q shape: torch.Size([2, 2])
Q value:
 tensor([[0.0010, 0.0000],
        [0.0000, 0.0010]])
R shape: torch.Size([2, 2])
R value:
 tensor([[0.1000, 0.0000],
        [0.0000, 0.1000]])


## Generation and loading of data

NOTE: For the Jacobian computation both analytical (computed with MATLAB symbolic toolbox) and numerical have been tested.

In [5]:
# Model Parameters
if SELECTED_MODEL == 0:
    # Synthetic NL model
    sys_model = SystemModel(f, Q, h, R, args.T, args.T_test, m, n)
    sys_model.InitSequence(m1x_0, m2x_0)# x0 and P0
    sys_model.alpha = 0.9
    sys_model.beta = 1.1
    sys_model.phi = 0.1*math.pi
    sys_model.delta = 0.01
    sys_model.a = 1
    sys_model.b = 1
    sys_model.c = 0
else:
    # Discrete Time Lorentz Attractor
    sys_model = SystemModel(f_nobatch, Q, h_nobatch, R, args.T, args.T_test, m, n)
    sys_model.InitSequence(m1x_0, m2x_0)

    # Model with partial info
    sys_model_partial = SystemModel(f_nobatch, Q, h_nobatch, R, args.T, args.T_test, m, n)
    sys_model_partial.InitSequence(m1x_0, m2x_0)

print("Start Data Gen")
DataGen(args, sys_model, DatafolderName + dataFileName[0])
print("Data Load")
print(dataFileName[0])
[train_input,train_target, _, _, _, _,_,_,_] =  torch.load(DatafolderName + dataFileName[0], map_location=device)
print("trainset input size:",train_input.size())
print("trainset target size:",train_target.size())

# Check of dimensions of inputs and outputs - Data Preprocessing
train_input = torch.squeeze(train_input) 
train_target = torch.squeeze(train_target)
print("trainset input size:",train_input.size())
print("trainset target size:",train_target.size())

Start Data Gen
Data Load
data_synNL_T100.pt
trainset input size: torch.Size([10, 2, 50])
trainset target size: torch.Size([10, 2, 50])
trainset input size: torch.Size([10, 2, 50])
trainset target size: torch.Size([10, 2, 50])


Plot of the test data

In [6]:
target = torch.Tensor.numpy(torch.transpose(train_target,0, 1))
inputs = torch.Tensor.numpy(torch.transpose(train_input,0, 1))

if SELECTED_MODEL == 0:
    fig = make_subplots(rows=2, cols=1, subplot_titles=("X[n]","Y[n]"))
    
    fig.append_trace(go.Scatter(x = np.linspace(1, target.shape[0], target.shape[0]+1),
        y=target[:,0], name = 'Target X_1'
    ), row=1, col=1)
    fig.append_trace(go.Scatter(x = np.linspace(1, target.shape[0], target.shape[0]+1),
        y=target[:,1],name = 'Target X_2'
    ), row=1, col=1)
    
    fig.append_trace(go.Scatter(x = np.linspace(1, inputs.shape[0], inputs.shape[0]+1),
        y=inputs[:,0],name = 'Y_1'
    ), row=2, col=1)
    fig.append_trace(go.Scatter(x = np.linspace(1, inputs.shape[0], inputs.shape[0]+1),
        y=inputs[:,1],name = 'Y_2'
    ), row=2, col=1)
    
    fig.show()
else:
    fig = make_subplots(rows=3, cols=1, subplot_titles=("X[n]","Y[n]"))
    
    fig.append_trace(go.Scatter(x = np.linspace(1, target.shape[0], target.shape[0]+1),
        y=target[:,0], name = 'Target X_1'
    ), row=1, col=1)
    fig.append_trace(go.Scatter(x = np.linspace(1, target.shape[0], target.shape[0]+1),
        y=target[:,1],name = 'Target X_2'
    ), row=1, col=1)
    fig.append_trace(go.Scatter(x = np.linspace(1, target.shape[0], target.shape[0]+1),
        y=target[:,2],name = 'Target X_3'
    ), row=1, col=1)
    
    fig.append_trace(go.Scatter(x = np.linspace(1, inputs.shape[0], inputs.shape[0]+1),
        y=inputs[:,0],name = 'Y_1'
    ), row=2, col=1)
    fig.append_trace(go.Scatter(x = np.linspace(1, inputs.shape[0], inputs.shape[0]+1),
        y=inputs[:,1],name = 'Y_2'
    ), row=2, col=1)
    fig.append_trace(go.Scatter(x = np.linspace(1, inputs.shape[0], inputs.shape[0]+1),
        y=inputs[:,2],name = 'Y_2'
    ), row=2, col=1)

    fig.append_trace(go.Scatter(x = target[:, 0],
        y=target[:, 2],name = 'X-Z plot of Lorentz Attractor'
    ), row=3, col=1)
    
    fig.show()

Generate multiple test sequences

In [7]:
num_of_test_sets = 100
test_inputs = []
test_targets = []

print(rf"## GENERATING TEST SEQUENCE OF {num_of_test_sets} SEQUENCES ##")
for i in range(num_of_test_sets):
    
    DataGen(args, sys_model, DatafolderName + dataFileName[0])
    [_, _, _, _, test_input, test_target, _, _, _] = torch.load(DatafolderName + dataFileName[0], map_location=device)

    test_inputs.append(test_input)
    test_targets.append(test_target)

## GENERATING TEST SEQUENCE OF 100 SEQUENCES ##


Create the dataframe for testing results

In [8]:
# Initialize dataframe for the test data
if SELECTED_MODEL == 0:
    # Synthetic NL Model
    test_data_synt = pd.DataFrame(columns=['MSE REKF','Computational Time REKF [s]','MSE Robust KNet','Computational Time Robust KNet [s]','MSE KalmanNet','Computational Time KalmanNet [s]'])
else:
    # Discrete Time Lorentz Attractor
    test_data_synt = pd.DataFrame(columns=['MSE REKF','Computational Time REKF [s]','MSE Robust KNet','Computational Time Robust KNet [s]','MSE KalmanNet','Computational Time KalmanNet [s]'])


In [9]:
# data check
print("test input size:",test_input.size())
print("test target size:",test_target.size())


test input size: torch.Size([5, 2, 50])
test target size: torch.Size([5, 2, 50])


## Test of REKF

In [10]:
print(rf"EVALUATING MSE AND COMPUTATIONAL TIME ON THE TEST SEQUENCES")

sys_model.m1x_0 = torch.zeros(m, 1)

sys_model.Q = Q
sys_model.R = R

c = 1e-3

i = 0
for test_input_batch, test_target_batch in zip(test_inputs, test_targets):
    # test_input_batch: [N_T, p, T]  es. [100, 2, 100]
    # Iterate over each sequence in the batch
    for seq_idx in range(test_input_batch.shape[0]):
        test_input = test_input_batch[seq_idx]   # [p, T]  es. [2, 100]
        test_target = test_target_batch[seq_idx]  # [n, T]  es. [2, 100]

        if SELECTED_MODEL == 0:
            REKF = RobustKalman(sys_model, test_input, c, True, False)
            [Xrekf, _, comp_time_rekf] = REKF.fnREKF()
            mse_rekf = mse_loss(Xrekf[:, :Xrekf.size()[1]-1], test_target)
            test_data_synt.loc[i, 'MSE REKF':'Computational Time REKF [s]'] = [mse_rekf.item(), comp_time_rekf]
            i += 1
        else:
            REKF = RobustKalman(sys_model, test_input, c, False, False, sl_model=SELECTED_MODEL)
            [Xrekf, _, comp_time_rekf] = REKF.fnREKF()
            mse_rekf = mse_loss(Xrekf[:, :Xrekf.size()[1]-1], test_target)
            test_data_synt.loc[i, 'MSE REKF':'Computational Time REKF [s]'] = [mse_rekf.item(), comp_time_rekf]
            i += 1
        
print("\n######   Test REKF   ######",f"\nAverage MSE: {test_data_synt.mean()['MSE REKF']:.5f}",f"\nAverage Computational Time [s]: {test_data_synt.mean()['Computational Time REKF [s]']:.5f}")

EVALUATING MSE AND COMPUTATIONAL TIME ON THE TEST SEQUENCES

######   Test REKF   ###### 
Average MSE: 0.02676 
Average Computational Time [s]: 0.18550


In [11]:
if SELECTED_MODEL == 0:
    
    REKFfig = make_subplots(rows=3, cols=1, subplot_titles=("X[n] - Only one sequence out of N","MSE[N]", "COMP_T[s][N]"))
    
    REKFfig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0], name = 'Target X_1'), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1], name = 'target X_2'), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 0], name = 'estimated X_1'), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 1], name = 'estimated X_2'), row=1, col=1)
    
    REKFfig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['MSE REKF'], name='MSE'), row=2, col=1)
    REKFfig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()[0], test_data_synt.mean()[0], len(test_data_synt)), name = 'MSE MEAN'), row=2, col=1)
    
    REKFfig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['Computational Time REKF [s]'], name='Comp. Time'), row=3, col=1)
    REKFfig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()[1], test_data_synt.mean()[1], len(test_data_synt)), name = 'Comp. Time MEAN'), row=3, col=1)
    
    REKFfig.show()
    
else:
    
    REKFfig = make_subplots(rows=1, cols=1, subplot_titles=("X[n] - Only one sequence out of N"))
    
    REKFfig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0].shape[0], torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0].shape[0]+1),
        y=torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0], name = 'target X_1'
    ), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1].shape[0], torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1].shape[0]+1),
        y=torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1], name = 'target X_2'
    ), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 2].shape[0], torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 2].shape[0]+1),
        y=torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 2], name = 'target X_3'
    ), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 0].shape[0], torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 0].shape[0]+1),
        y=torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 0], name = 'estimated X_1'
    ), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 1].shape[0], torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 1].shape[0]+1),
        y=torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 1], name = 'estimated X_2'
    ), row=1, col=1)
    REKFfig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 2].shape[0], torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 2].shape[0]+1),
        y=torch.squeeze(torch.transpose(Xrekf,0, 1)).detach().numpy()[0:-1, 2], name = 'estimated X_3'
    ), row=1, col=1)

    REKFfig.show()

    Lorentz = make_subplots(rows=1, cols=1, subplot_titles=("Attractor phase space"))
    Lorentz.append_trace(go.Scatter(x = test_target[0,:], y= test_target[2, :], name="Target"), row=1, col=1)
    Lorentz.append_trace(go.Scatter(x = Xrekf[0,:].detach(), y= Xrekf[2, :].detach(), name="Estimated"), row=1, col=1)
    
    Lorentz.show()

## Robust-KalmanNet

Training phase

In [12]:
import torch.optim as optim
import torch.nn as nn
from tqdm.notebook import tqdm  # Use tqdm notebook version for Jupyter

sys_model.m1x_0 = torch.zeros(m, 1)

# Select the number of hidden-layers and neurons in the DNN
gru_hidden_size = 64  # Puoi sperimentare con 32, 64 o 128

# Initialize the noise covariance matrices Q and R for the filter
Q_2 = torch.eye(sys_model.m)
R_2 = torch.eye(sys_model.n)

# Create the Robust KalmanNet instance
RT_KalmanNet = RobustKalman(
    sys_model, 
    train_input, 
    c, 
    False, 
    True, 
    input_feat_mode=3, 
    gru_hidden_size=gru_hidden_size, # <-- IL NUOVO PARAMETRO
    sl_model=SELECTED_MODEL, 
    set_noise_matrices=False, 
    Q_mat=Q_2, 
    R_mat=R_2
)

# Hyperparameters
epochs = args.n_steps  # Number of epochs

train_inputs = []
train_targets = []
cv_inputs = []
cv_targets = []

# Generate training and cross-validation datasets
print(f"GENERATING TRAINING AND CROSS_VALIDATION DATA")
for epoch in range(epochs):
    
    DataGen(args, sys_model, DatafolderName + dataFileName[0])
    [train_input, train_target, cv_input, cv_target, _, _, _, _, _] = torch.load(DatafolderName + dataFileName[0], map_location=device)

    train_inputs.append(train_input)
    train_targets.append(train_target)
    cv_inputs.append(cv_input)
    cv_targets.append(cv_target)


GENERATING TRAINING AND CROSS_VALIDATION DATA


In [13]:
# data check
print("trainset input size:",train_input.size())
print("trainset target size:",train_target.size())
print("cv input size:",cv_input.size())
print("cv target size:",cv_target.size())

trainset input size: torch.Size([10, 2, 50])
trainset target size: torch.Size([10, 2, 50])
cv input size: torch.Size([5, 2, 50])
cv target size: torch.Size([5, 2, 50])


In [14]:
import random
import matplotlib.pyplot as plt

# Set optimizer and loss function type
optimizer = optim.Adam(RT_KalmanNet.nn.parameters(), lr=args.lr, weight_decay=args.wd)
criterion = nn.MSELoss(reduction='mean')

opt_MSE = float('inf')
best_model_path = 'RobustKalmanPY/opt_RT_KNet.pt'

# ===== Lists to track losses =====
train_losses = []
cv_losses = []

for epoch in range(epochs):
    # ===== Train =====
    optimizer.zero_grad(set_to_none=True)

    train_batch = train_inputs[epoch]          # [N_E, p, T]
    train_target_batch = train_targets[epoch]  # [N_E, n, T]
    
    # Sample n_batch indices from training dataset
    n_available = train_batch.shape[0]
    batch_indices = random.sample(range(n_available), min(args.n_batch, n_available))
    
    batch_losses = []
    for idx in batch_indices:
        # Reset filter state for each sequence
        RT_KalmanNet.Xrekf_prev = RT_KalmanNet.x0.squeeze(0).detach().clone()
        RT_KalmanNet.y_prev = torch.zeros(RT_KalmanNet.p, device=RT_KalmanNet.Xrekf_prev.device)
        RT_KalmanNet.Xn_prev = torch.zeros(RT_KalmanNet.n, device=RT_KalmanNet.Xrekf_prev.device)
        RT_KalmanNet.V_prev = (1e-3 * torch.eye(RT_KalmanNet.n, device=RT_KalmanNet.Xrekf_prev.device)).detach()

        # Get single sequence from batch
        RT_KalmanNet.y = train_batch[idx]  # [p, T]
        Xrekf, _, _, _ = RT_KalmanNet.fnREKF(train=True)
        Xrekf = Xrekf[:, :-1]  # [n, T]

        train_target = train_target_batch[idx]  # [n, T]
        loss = criterion(Xrekf, train_target)
        batch_losses.append(loss)
    
    # Average loss over batch and backward
    avg_loss = torch.stack(batch_losses).mean()
    avg_loss.backward()
    optimizer.step()
    
    # Save train loss
    train_losses.append(avg_loss.item())

    # ===== CV =====
    with torch.no_grad():
        cv_batch = cv_inputs[epoch]          # [N_CV, p, T]
        cv_target_batch = cv_targets[epoch]  # [N_CV, n, T]
        
        cv_losses_batch = []
        for idx in range(cv_batch.shape[0]):
            # Reset filter state for each sequence
            RT_KalmanNet.Xrekf_prev = RT_KalmanNet.x0.squeeze(0).detach().clone()
            RT_KalmanNet.y_prev = torch.zeros(RT_KalmanNet.p, device=RT_KalmanNet.Xrekf_prev.device)
            RT_KalmanNet.Xn_prev = torch.zeros(RT_KalmanNet.n, device=RT_KalmanNet.Xrekf_prev.device)
            RT_KalmanNet.V_prev = (1e-3 * torch.eye(RT_KalmanNet.n, device=RT_KalmanNet.Xrekf_prev.device)).detach()

            RT_KalmanNet.y = cv_batch[idx]  # [p, T]
            Xrekf_cv, _, _, _ = RT_KalmanNet.fnREKF(train=False)
            Xrekf_cv = Xrekf_cv[:, :-1]  # [n, T]

            cv_target = cv_target_batch[idx]  # [n, T]
            cv_loss = criterion(Xrekf_cv, cv_target)
            cv_losses_batch.append(cv_loss.item())
        
        avg_cv_loss = np.mean(cv_losses_batch)
        cv_losses.append(avg_cv_loss)

        if avg_cv_loss < opt_MSE:
            opt_MSE = avg_cv_loss
            torch.save(RT_KalmanNet.nn.state_dict(), best_model_path)

    if (epoch + 1) % 10 == 0:  # Print every 10 epochs to avoid spam
        tqdm.write(f"Epoch {epoch+1}/{epochs}, MSE Training: {train_losses[-1]:.6f}, MSE CV: {avg_cv_loss:.6f}")

print("\nTraining finished")
print(f"Cross-Validation MSE Optimal Model: {opt_MSE:.4f}\n")

# ===== Plot Training and Validation Losses =====
fig = make_subplots(rows=1, cols=1, subplot_titles=("Training and Validation Loss"))

fig.add_trace(
    go.Scatter(x=list(range(1, len(train_losses)+1)), y=train_losses, name='Training Loss', mode='lines'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=list(range(1, len(cv_losses)+1)), y=cv_losses, name='Validation Loss', mode='lines'),
    row=1, col=1
)

fig.update_xaxes(title_text="Epoch", row=1, col=1)
fig.update_yaxes(title_text="MSE Loss", row=1, col=1)
fig.update_layout(hovermode='x unified', title_text="RT-KalmanNet Training Progress", height=500)
fig.show()

# Optional: Print final statistics
print(f"\nFinal Train Loss: {train_losses[-1]:.6f}")
print(f"Final CV Loss: {cv_losses[-1]:.6f}")
print(f"Min CV Loss: {min(cv_losses):.6f} at epoch {cv_losses.index(min(cv_losses))+1}")

Epoch 10/200, MSE Training: 0.004751, MSE CV: 0.004665
Epoch 20/200, MSE Training: 0.005158, MSE CV: 0.005207
Epoch 30/200, MSE Training: 0.005594, MSE CV: 0.004563
Epoch 40/200, MSE Training: 0.004659, MSE CV: 0.004583
Epoch 50/200, MSE Training: 0.004597, MSE CV: 0.004279
Epoch 60/200, MSE Training: 0.004822, MSE CV: 0.004695
Epoch 70/200, MSE Training: 0.005664, MSE CV: 0.004567
Epoch 80/200, MSE Training: 0.005555, MSE CV: 0.004773
Epoch 90/200, MSE Training: 0.005415, MSE CV: 0.004975
Epoch 100/200, MSE Training: 0.005425, MSE CV: 0.004581
Epoch 110/200, MSE Training: 0.004455, MSE CV: 0.004576
Epoch 120/200, MSE Training: 0.005074, MSE CV: 0.005455
Epoch 130/200, MSE Training: 0.004727, MSE CV: 0.004330
Epoch 140/200, MSE Training: 0.004958, MSE CV: 0.004685
Epoch 150/200, MSE Training: 0.004737, MSE CV: 0.004731
Epoch 160/200, MSE Training: 0.005145, MSE CV: 0.004510
Epoch 170/200, MSE Training: 0.005663, MSE CV: 0.004522
Epoch 180/200, MSE Training: 0.004669, MSE CV: 0.004787
E


Final Train Loss: 0.004687
Final CV Loss: 0.004603
Min CV Loss: 0.003852 at epoch 41


Test of Robust-KalmanNet - Computational time and RMSE

In [15]:
# Load the optimal model from cross validation procedure
RT_KalmanNet.nn.load_state_dict(torch.load(best_model_path))
RT_KalmanNet.nn.eval()

# Test over each time series
i = 0
print(rf"## EVALUATING MSE AND COMPUTATIONAL TIME ON THE TEST SEQUENCES ##")

with torch.no_grad():
    for test_input_batch, test_target_batch in zip(test_inputs, test_targets):
        # test_input_batch: [N_T, p, T]
        # Iterate over each sequence in the batch
        for seq_idx in range(test_input_batch.shape[0]):
            test_input = test_input_batch[seq_idx]   # [p, T]
            test_target = test_target_batch[seq_idx]  # [n, T]

            # Reset filter state for test
            RT_KalmanNet.Xrekf_prev = RT_KalmanNet.x0.squeeze(0).detach().clone()
            RT_KalmanNet.y_prev = torch.zeros(RT_KalmanNet.p, device=RT_KalmanNet.Xrekf_prev.device)
            RT_KalmanNet.Xn_prev = torch.zeros(RT_KalmanNet.n, device=RT_KalmanNet.Xrekf_prev.device)
            RT_KalmanNet.V_prev = (1e-3 * torch.eye(RT_KalmanNet.n, device=RT_KalmanNet.Xrekf_prev.device)).detach()

            # Compute the RT-KalmanNet prediction
            RT_KalmanNet.y = test_input  # [p, T]
            [Xrekf, c_t, _, comp_time_RT_KNet] = RT_KalmanNet.fnREKF()
            Xrekf = Xrekf[:, :-1].detach()  # [n, T]

            test_loss = criterion(Xrekf, test_target)

            # Save MSE and Computational time values on dataframe
            test_data_synt.loc[i, 'MSE Robust KNet':'Computational Time Robust KNet [s]'] = [test_loss.item(), comp_time_RT_KNet]
            i += 1

print("#####  Test RT-KalmanNet  #####",f"\nMSE: {test_data_synt.mean()['MSE Robust KNet']:.4f}",f"\nComputational Time: {test_data_synt.mean()['Computational Time Robust KNet [s]']:.4f}")

## EVALUATING MSE AND COMPUTATIONAL TIME ON THE TEST SEQUENCES ##
#####  Test RT-KalmanNet  ##### 
MSE: 0.0266 
Computational Time: 0.1602


In [16]:
RobustKNetFig = make_subplots(rows=3, cols=1, subplot_titles=("X[n] - Only one sequence out of N","MSE[N]", "COMP_T[s][N]", "Tollerance C_t"))

if SELECTED_MODEL == 0:
    RobustKNetFig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0], name = 'target X_1'), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1], name = 'target X_2'), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 0], name = 'estimated X_1'), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.arange(len(test_data_synt)), y = torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 1], name = 'estimated X_2'), row=1, col=1)

    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['MSE Robust KNet'], name='MSE'), row=2, col=1)
    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()[2], test_data_synt.mean()[2], len(test_data_synt)), name = 'MSE MEAN'), row=2, col=1)
    
    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['Computational Time Robust KNet [s]'], name='Comp. Time'), row=3, col=1)
    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()[3], test_data_synt.mean()[3], len(test_data_synt)), name = 'Comp. Time MEAN'), row=3, col=1)
    
    RobustKNetFig.show()
else:
    RobustKNetFig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0].shape[0], torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0].shape[0]+1),
        y=torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 0], name = 'target X_1'
    ), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1].shape[0], torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1].shape[0]+1),
        y=torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1], name = 'target X_2'
    ), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1].shape[0], torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 1].shape[0]+1),
        y=torch.squeeze(torch.transpose(test_target,0, 1)).numpy()[0:-1, 2], name = 'target X_3'
    ), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 0].shape[0], torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 0].shape[0]+1),
        y=torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 0], name = 'estimated X_1'
    ), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 1].shape[0], torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 1].shape[0]+1),
        y=torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 1], name = 'estimated X_2'
    ), row=1, col=1)
    RobustKNetFig.append_trace(go.Scatter(x = np.linspace(1, torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 2].shape[0], torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 2].shape[0]+1),
        y=torch.squeeze(torch.transpose(Xrekf.detach(), 0, 1)).numpy()[0:-1, 2], name = 'estimated X_3'
    ), row=1, col=1)

    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['MSE Robust KNet'], name='MSE'), row=2, col=1)
    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()[2], test_data_synt.mean()[2], len(test_data_synt)), name = 'MSE MEAN'), row=2, col=1)

    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['Computational Time Robust KNet [s]'], name='Comp. Time'), row=3, col=1)
    RobustKNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()[3], test_data_synt.mean()[3], len(test_data_synt)), name = 'Comp. Time MEAN'), row=3, col=1)
    
    RobustKNetFig.show()

    Lorentz = make_subplots(rows=1, cols=1, subplot_titles=("A sample plot of the test data", "A sample plot of the filtered data"))
    Lorentz.append_trace(go.Scatter(x = test_target[0,:], y= test_target[2, :]), row=1, col=1)
    Lorentz.append_trace(go.Scatter(x = Xrekf[0,:], y= Xrekf[2, :]), row=1, col=1)
    
    Lorentz.show()

In [17]:
# Plot tollerance c_t
# c_t è una lista di tensor (uno per time-step)
c_t_np = torch.stack([ct.detach() for ct in c_t]).cpu().numpy().reshape(-1)

c_tFig = go.Figure()
c_tFig.add_trace(go.Scatter(x=np.arange(c_t_np.shape[0]), y=c_t_np, name="c_t"))
c_tFig.update_layout(
    title=dict(text='Tolerance c_t', font=dict(size=20), x=0.5, xanchor='center'),
    xaxis=dict(title='Time (s)', showgrid=True, gridcolor='lightgray', zeroline=False),
    yaxis=dict(showgrid=True, gridcolor='lightgray', zeroline=False),
    template='simple_white'
)
c_tFig.show()

## KalmanNet Test

In [18]:
# Use same dataset size as RT-KalmanNet for fair comparison
args.N_E = 200
args.N_CV = 50
args.N_T = 100
args.n_batch = 10

today = datetime.today()
now = datetime.now()
strToday = today.strftime("%m.%d.%y")
strNow = now.strftime("%H:%M:%S")
strTime = strToday + "_" + strNow

if SELECTED_MODEL == 0:
    DataGen(args, sys_model, DatafolderName + dataFileName[0])
    [train_input, train_target, cv_input, cv_target, test_input, test_target, _, _, _] = torch.load(DatafolderName + dataFileName[0], map_location=device)

    ## Build Neural Network
    KalmanNet_model = KalmanNetNN()
    KalmanNet_model.NNBuild(sys_model, args)

    ## Train Neural Network
    print("#####   Training KalmanNet   #####\n")
    KalmanNet_Pipeline = Pipeline_EKF(strTime, "KNet", "KalmanNet")
    KalmanNet_Pipeline.setssModel(sys_model)
    KalmanNet_Pipeline.setModel(KalmanNet_model)
    print("Number of trainable parameters for KNet:", sum(p.numel() for p in KalmanNet_model.parameters() if p.requires_grad))
    KalmanNet_Pipeline.setTrainingParams(args)

    [MSE_cv_linear_epoch, MSE_cv_dB_epoch, MSE_train_linear_epoch, MSE_train_dB_epoch] = KalmanNet_Pipeline.NNTrain(sys_model, cv_input, cv_target, train_input, train_target, path_results)

    # Test evaluation - same test sets as REKF and RT-KNet for fair comparison
    i = 0
    for test_input_batch, test_target_batch in zip(test_inputs, test_targets):
        for seq_idx in range(test_input_batch.shape[0]):
            seq_input = test_input_batch[seq_idx].unsqueeze(0)   # [1, n, T]
            seq_target = test_target_batch[seq_idx].unsqueeze(0)  # [1, m, T]

            [MSE_test_linear_arr, MSE_test_linear_avg, MSE_test_dB_avg, knet_out, comp_time_KNet] = KalmanNet_Pipeline.NNTest(sys_model, seq_input, seq_target, path_results)
            test_data_synt.loc[i, 'MSE KalmanNet':'Computational Time KalmanNet [s]'] = [MSE_test_linear_avg.item(), comp_time_KNet]
            i += 1
else:
    DatafolderName = 'Simulations/Lorenz_Atractor/data' + '/'
    traj_resultName = ['traj_lorDT_rq1030_T100.pt']
    dataFileName = ['data_lor_v20_rq1030_T100.pt']

    sys_model = SystemModel(f, Q, hRotate, R, args.T, args.T_test, m, n)
    sys_model.InitSequence(m1x_0, m2x_0)

    DataGen(args, sys_model, DatafolderName + dataFileName[0])
    [train_input_long, train_target_long, cv_input, cv_target, test_input, test_target, _, _, _] = torch.load(DatafolderName + dataFileName[0], map_location=device)
    train_target = train_target_long[:, :, 0:args.T]
    train_input = train_input_long[:, :, 0:args.T]

    KNet_model = KalmanNetNN()
    KNet_model.NNBuild(sys_model, args)

    print("#####   Training KalmanNet   #####\n")
    KNet_Pipeline = Pipeline_EKF(strTime, "KNet", "KNet")
    KNet_Pipeline.setssModel(sys_model)
    KNet_Pipeline.setModel(KNet_model)
    print("Number of trainable parameters for KNet:", sum(p.numel() for p in KNet_model.parameters() if p.requires_grad))
    KNet_Pipeline.setTrainingParams(args)

    [MSE_cv_linear_epoch, MSE_cv_dB_epoch, MSE_train_linear_epoch, MSE_train_dB_epoch] = KNet_Pipeline.NNTrain(sys_model, cv_input, cv_target, train_input, train_target, path_results)

        # Test evaluation - same test sets as REKF and RT-KNet for fair comparison
    i = 0
    for test_input_batch, test_target_batch in zip(test_inputs, test_targets):
        for seq_idx in range(test_input_batch.shape[0]):
            seq_input = test_input_batch[seq_idx].unsqueeze(0)   # [1, n, T]
            seq_target = test_target_batch[seq_idx].unsqueeze(0)  # [1, m, T]

            [MSE_test_linear_arr, MSE_test_linear_avg, MSE_test_dB_avg, knet_out, comp_time_KNet] = KalmanNet_Pipeline.NNTest(sys_model, seq_input, seq_target, path_results)
            test_data_synt.loc[i, 'MSE KalmanNet':'Computational Time KalmanNet [s]'] = [MSE_test_linear_avg.item(), comp_time_KNet]
            i += 1

print("\n#####  Test KalmanNet  #####", f"\nMSE: {test_data_synt.mean()['MSE KalmanNet']:.4f}", f"\nComputational Time: {test_data_synt.mean()['Computational Time KalmanNet [s]']:.4f}")

#####   Training KalmanNet   #####

Number of trainable parameters for KNet: 5208
Epoch 1/200, MSE Training: 0.0020, MSE Validation: 0.0014
Epoch 2/200, MSE Training: 0.0014, MSE Validation: 0.0013
Epoch 3/200, MSE Training: 0.0014, MSE Validation: 0.0014
Epoch 4/200, MSE Training: 0.0014, MSE Validation: 0.0014
Epoch 5/200, MSE Training: 0.0014, MSE Validation: 0.0014
Epoch 6/200, MSE Training: 0.0013, MSE Validation: 0.0013
Epoch 7/200, MSE Training: 0.0013, MSE Validation: 0.0012
Epoch 8/200, MSE Training: 0.0011, MSE Validation: 0.0012
Epoch 9/200, MSE Training: 0.0011, MSE Validation: 0.0012
Epoch 10/200, MSE Training: 0.0012, MSE Validation: 0.0012
Epoch 11/200, MSE Training: 0.0012, MSE Validation: 0.0012
Epoch 12/200, MSE Training: 0.0013, MSE Validation: 0.0012
Epoch 13/200, MSE Training: 0.0013, MSE Validation: 0.0012
Epoch 14/200, MSE Training: 0.0011, MSE Validation: 0.0011
Epoch 15/200, MSE Training: 0.0011, MSE Validation: 0.0011
Epoch 16/200, MSE Training: 0.0010, MSE Va

In [19]:
KNetFig = make_subplots(rows=3, cols=1, subplot_titles=("X[n] - Only one sequence out of N", "MSE[N]", "COMP_T[s][N]"))

KNetFig.append_trace(go.Scatter(x=np.arange(test_target.detach().size(2)),y=test_target.detach().numpy()[0,0,0:-1], name = 'Target X_1'), row=1, col=1)
KNetFig.append_trace(go.Scatter(x=np.arange(test_target.detach().size(2)),y=test_target.detach().numpy()[0,1,0:-1], name = 'Target X_2'), row=1, col=1)
if SELECTED_MODEL == 1:
    KNetFig.append_trace(go.Scatter(x=np.arange(knet_out.detach().size(2)),y=test_target.detach().numpy()[0,2,0:-1], name = 'Target X_3'), row=1, col=1)
KNetFig.append_trace(go.Scatter(x=np.arange(knet_out.detach().size(2)),y=knet_out.detach().numpy()[0,0,0:-1], name = 'Estimated X_1'), row=1, col=1)
KNetFig.append_trace(go.Scatter(x=np.arange(knet_out.detach().size(2)),y=knet_out.detach().numpy()[0,1,0:-1], name = 'Estimated X_2'), row=1, col=1)
if SELECTED_MODEL == 1:
    KNetFig.append_trace(go.Scatter(x=np.arange(knet_out.detach().size(2)),y=knet_out.detach().numpy()[0,2,0:-1], name = 'Estimated X_3'), row=1, col=1)

KNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['MSE KalmanNet'], name='MSE'), row=2, col=1)
KNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()['MSE KalmanNet'], test_data_synt.mean()['MSE KalmanNet'], len(test_data_synt)), name = 'MSE MEAN'), row=2, col=1)

KNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=test_data_synt['Computational Time KalmanNet [s]'], name='Comp. Time'), row=3, col=1)
KNetFig.append_trace(go.Scatter(x=np.arange(len(test_data_synt)), y=np.linspace(test_data_synt.mean()['Computational Time KalmanNet [s]'], test_data_synt.mean()['Computational Time KalmanNet [s]'], len(test_data_synt)), name = 'Comp. Time MEAN'), row=3, col=1)

KNetFig.show()
if SELECTED_MODEL == 1:
    Lorentz = make_subplots(rows=1, cols=1, subplot_titles=("A sample plot of the test data", "A sample plot of the filtered data"))
    Lorentz.append_trace(go.Scatter(x = test_target.detach().numpy()[0,0,0:-1], y= test_target.detach().numpy()[0,2,0:-1]), row=1, col=1)
    Lorentz.append_trace(go.Scatter(x = knet_out.detach().numpy()[0,0,0:-1], y= knet_out.detach().numpy()[0,2,0:-1]), row=1, col=1)
    
    Lorentz.show()

### Statistics of MSE and Computational Time for the Synthetic Non-Linear Model

In [20]:
stats = pd.DataFrame({
    'MEAN': test_data_synt.mean(),
    'STD': test_data_synt.std()
})

print(stats.applymap(lambda x: f"{x:.6f}"))

                                        MEAN       STD
MSE REKF                            0.026763  0.001193
Computational Time REKF [s]         0.185504  0.005899
MSE Robust KNet                     0.026589  0.001182
Computational Time Robust KNet [s]  0.160165  0.006371
MSE KalmanNet                       0.010827  0.000708
Computational Time KalmanNet [s]    0.012099  0.002879
